# TP3 — RAG Systems and Vector Databases

## Objectives
- Represent text as vectors (Bag of Words, Transformers, OpenAI embeddings).
- Measure semantic similarity with cosine similarity.
- Build a RAG pipeline with FAISS and LangChain.
- Apply text splitting and explore different loaders.


## Setup


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
print('Keys loaded:', bool(os.getenv('MISTRAL_API_KEY')))


## Step 2 — Bag of Words (Ex 2.1, 2.2)


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    'Demonstration text, first document',
    "Demo text, and here's a second document.",
    'And finally, this is the third document.',
    'This showcase illustrates a fourth document example.',  # Ex 2.1: synonym 'showcase'
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

print('Vocabulary:', vectorizer.get_feature_names_out())
print('BoW matrix:\n', X.toarray())
print()
# Ex 2.1: 'showcase' is a new column — BoW cannot relate it to 'demo'
# Ex 2.2 limitations:
# 1. No synonym awareness: 'demo' != 'showcase'
# 2. Word order ignored: 'dog bites man' == 'man bites dog'
print('Limitations: no synonyms, no word order.')


## Step 4 — Cosine Similarity (Ex 4.1, 4.2)


In [ ]:
import numpy as np

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

vectors = X.toarray().astype(float)
print('Similarity(doc0, doc1):', cosine_similarity(vectors[0], vectors[1]))

# Ex 4.1: very different docs -> low similarity
print('Similarity(doc0, doc3):', cosine_similarity(vectors[0], vectors[3]))

# Ex 4.2: nearest-neighbour function
def find_most_similar(query, corpus_vectors, vec):
    q = vec.transform([query]).toarray().astype(float)[0]
    sims = [cosine_similarity(q, d) for d in corpus_vectors]
    return int(np.argmax(sims)), sims

idx, sims = find_most_similar('first demonstration document', vectors, vectorizer)
print(f'Best match: doc{idx} (score={sims[idx]:.4f})')


## Step 3 — Sentence Transformers Embeddings (Ex 3.1, 3.2)


In [ ]:
from sentence_transformers import SentenceTransformer

sentences = [
    'This is an example sentence.',
    'Each sentence is converted into a fixed-sized vector.',
]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')
embeddings = model.encode(sentences)

for s, e in zip(sentences, embeddings):
    print(f'{s!r} -> first 3 dims: {e[:3]}  size: {len(e)}')

# Ex 3.1: compare model sizes
for model_name in [
    'sentence-transformers/all-MiniLM-L6-v2',
    'sentence-transformers/all-MiniLM-L12-v2',
    'sentence-transformers/all-distilroberta-v1',
]:
    m = SentenceTransformer(model_name, device='cpu')
    e = m.encode(sentences[0])
    print(f'{model_name}: size={len(e)}')

# Ex 3.2: device='cpu' avoids CUDA errors on machines without a GPU
print('device=cpu ensures reproducibility across all machines.')


## Step 6 — Basic RAG Pipeline (Ex 6.1, 6.2)


In [ ]:
from langchain_mistralai.chat_models import ChatMistralAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatMistralAI(model='mistral-large-latest')

query = 'What is Mycobacterium kansasii?'
context = (
    'Mycobacterium kansasii is a slow-growing, photochromogenic non-tuberculous '
    'mycobacterium that can cause pulmonary and extra-pulmonary infections. '
    'It is typically treated with rifampicin, ethambutol and clarithromycin.'
)

# Base English prompt with out-of-context guard (Ex 6.2)
prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'You are a Mycobacterium expert. Answer using ONLY the context. '
        "If the context does not contain the answer, reply: 'I don't know.'"
    )),
    ('human', 'Question: {query}\n\nContext:\n{context}'),
])
chain = prompt | llm
print(chain.invoke({'query': query, 'context': context}).content)

# Ex 6.1: force French
prompt_fr = ChatPromptTemplate.from_messages([
    ('system', 'Tu es un expert. R\u00e9ponds en fran\u00e7ais en utilisant UNIQUEMENT le contexte.'),
    ('human', 'Question : {query}\n\nContexte :\n{context}'),
])
print(prompt_fr.pipe(llm).invoke({'query': query, 'context': context}).content)

# Ex 6.2: out-of-context query
oos = chain.invoke({'query': 'What is the GDP of France?', 'context': context})
print('Out-of-scope answer:', oos.content)


## Step 7 — FAISS Vector Store + Step 9 Text Splitting (Ex 7.1, 7.2, 9.1, 9.2)

> Place `Guyeux_2024.pdf` in a `images/` subfolder relative to this notebook.


In [ ]:
import warnings
import time
from pathlib import Path
from textwrap import fill, shorten

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_mistralai.chat_models import ChatMistralAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

warnings.simplefilter('ignore')

PDF_PATH = Path('images/Guyeux_2024.pdf')
if not PDF_PATH.exists():
    print('PDF not found — place Guyeux_2024.pdf in images/ to run this cell.')
else:
    embed_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    pages = PyPDFLoader(str(PDF_PATH)).load_and_split()
    print(f'Loaded {len(pages)} pages.')

    t0 = time.perf_counter()
    index = FAISS.from_documents(pages, embed_model)
    print(f'Indexed in {time.perf_counter()-t0:.2f}s')

    # Base query
    docs = index.similarity_search('Is there a lineage 10 in M.tuberculosis?', k=2)
    for doc in docs:
        print(f'Page {doc.metadata.get("page","?")}: {fill(shorten(doc.page_content,400),80)}')

    # Ex 7.1: geographic query
    docs_geo = index.similarity_search('Geographic location of lineage 2 M.tuberculosis', k=2)
    for doc in docs_geo:
        print(f'Page {doc.metadata.get("page","?")}: {fill(shorten(doc.page_content,400),80)}')

    # Ex 7.2: full RAG pipeline
    llm = ChatMistralAI(model='mistral-large-latest')
    rag_prompt = ChatPromptTemplate.from_messages([
        ('system', 'Answer using ONLY the context. If unsure, say I don\'t know.'),
        ('human', 'Question: {query}\n\nContext:\n{context}'),
    ])
    def rag(q):
        ctx = '\n\n'.join(d.page_content for d in index.similarity_search(q, k=3))
        return (rag_prompt | llm).invoke({'query': q, 'context': ctx}).content
    print(rag('Is there a lineage 10 in M.tuberculosis?'))

    # Ex 9.1-9.2: text splitting
    for cfg in [{'chunk_size':1000,'chunk_overlap':200}, {'chunk_size':500,'chunk_overlap':100}]:
        splitter = RecursiveCharacterTextSplitter(**cfg, separators=['\n\n','\n','. '])
        chunks = splitter.split_documents(pages)
        print(f'chunk_size={cfg["chunk_size"]} \u2192 {len(chunks)} chunks')


## Step 10 — Other Loaders (Ex 10.1, 10.2)


In [ ]:
# Ex 10.1: useful loaders
loaders = [
    ('UnstructuredMarkdownLoader', 'Load .md files (docs, READMEs, wikis)'),
    ('BSHTMLLoader',               'Parse HTML pages'),
    ('CSVLoader',                  'Load tabular CSV data'),
    ('GitLoader',                  'Index source code from a Git repository'),
    ('NotionDirectoryLoader',      'Load exported Notion pages'),
]
for name, use in loaders:
    print(f'{name}: {use}')

# Ex 10.2: vector store comparison
print()
comparison = [
    ('FAISS',    'Fast local search, no persistence, no filtering, single machine'),
    ('Chroma',   'Persistent, metadata filtering, easy local dev'),
    ('Milvus',   'Distributed, scalable, hybrid search, production-grade'),
    ('Weaviate', 'GraphQL API, hybrid keyword+semantic, strong multi-tenancy'),
]
for name, desc in comparison:
    print(f'{name}: {desc}')
